# Canaux de communication multi-agents — le bus réel, mesuré (CoursIA)

Les huit modules de `argumentation_analysis/core/communication/` distillés en un
support prêt à l'emploi : un format de message commun, un contrat de canal,
quatre familles de canaux réelles (hiérarchique, collaboration, données,
pub/sub), un protocole requête-réponse, et le middleware qui route le tout.
Tout ce que ce notebook affiche vient du **vrai moteur** — chaque cellule
recharge les exemples depuis `docs/coursia_contrib/communication_channels_examples.json`,
source de vérité unique partagée avec la garde
`tests/unit/coursia/communication_channels/`, et **assertionne** que la valeur
attendue est la valeur mesurée.

**Déterministe et sûr** : aucun corpus, aucun appel LLM, aucune JVM. Les
horodatages sont fixés, les canaux et groupes portent des noms d'emprunt. Les
protocoles qui spawnent des threads sont arrêtés proprement en fin de notebook.

In [1]:
# Imports + inventaire mesuré. Aucun LLM, aucun corpus, aucune JVM.
import json
import logging
import time
from datetime import datetime, timedelta

logging.getLogger().setLevel(logging.ERROR)  # garder la sortie lisible

from argumentation_analysis.core.communication.channel_interface import (
    Channel, ChannelType, LocalChannel,
)
from argumentation_analysis.core.communication.collaboration_channel import CollaborationChannel
from argumentation_analysis.core.communication.data_channel import DataChannel
from argumentation_analysis.core.communication.hierarchical_channel import HierarchicalChannel
from argumentation_analysis.core.communication.message import (
    AgentLevel, CommandMessage, EventMessage, InformationMessage, Message,
    MessagePriority, MessageType,
)
from argumentation_analysis.core.communication.middleware import (
    MessageMiddleware, create_default_middleware,
)
from argumentation_analysis.core.communication.pub_sub import (
    PublishSubscribeProtocol, Topic,
)
from argumentation_analysis.core.communication.request_response import (
    RequestResponseProtocol, RequestTimeoutError,
)

EXAMPLES = json.load(open("docs/coursia_contrib/communication_channels_examples.json", encoding="utf-8"))
INV = EXAMPLES["inventory"]

T0 = datetime(2026, 1, 1, 12, 0, 0)


def msg(mtype=MessageType.INFORMATION, sender="agent-a", level=AgentLevel.TACTICAL,
        priority=MessagePriority.NORMAL, content=None, recipient=None, channel=None, ts=None,
        metadata=None, mid=None):
    return Message(message_type=mtype, sender=sender, sender_level=level,
                   content=content or {}, recipient=recipient, channel=channel,
                   priority=priority, metadata=metadata, message_id=mid, timestamp=ts or T0)


assert INV["message_types"] == [t.value for t in MessageType]
print("MessageType   :", len(MessageType), "valeurs")
print("MessagePriority:", len(MessagePriority), "| AgentLevel:", len(AgentLevel), "| ChannelType:", len(ChannelType))
print("Méthodes abstraites du canal :", INV["channel_abstract_methods"])
print("Clés de filtre honorées (#2161) :", Channel.FILTER_KEYS)

MessageType   : 8 valeurs
MessagePriority: 4 | AgentLevel: 4 | ChannelType: 7
Méthodes abstraites du canal : ['get_channel_info', 'get_pending_messages', 'receive_message', 'send_message', 'subscribe', 'unsubscribe']
Clés de filtre honorées (#2161) : frozenset({'sender', 'priority', 'message_type', 'sender_level', 'content'})


## 1. Le format message — priorité INVERSÉE, identité dérivable

Un `Message` porte type / émetteur / niveau / priorité / contenu, un id dérivé
du type (`command-<hex8>`), et un **ordonnancement total inversé** : `__lt__`
est écrit pour que la priorité la plus HAUTE soit la plus « petite » —
`sorted()` sort donc CRITICAL en premier, même arrivé en dernier. À priorité
égale, le plus ancien passe d'abord.

In [2]:
for case in EXAMPLES["message_cases"]:
    s = case["scenario"]
    if s == "id_prefix":
        m = msg(MessageType.COMMAND)
        assert m.id.split("-")[0] == case["expected"]
        print(f"id du message COMMAND : {m.id}  (préfixe « {case['expected']} »)")
    elif s == "priority_order_inverted_timestamps":
        msgs = [
            msg(priority=MessagePriority.LOW, ts=T0),
            msg(priority=MessagePriority.NORMAL, ts=T0 + timedelta(seconds=10)),
            msg(priority=MessagePriority.HIGH, ts=T0 + timedelta(seconds=20)),
            msg(priority=MessagePriority.CRITICAL, ts=T0 + timedelta(seconds=30)),
        ]
        got = [m.priority.value for m in sorted(msgs)]
        assert got == case["expected"], (got, case["expected"])
        print("sorted() avec horodatages INVERSÉS :", got)
    elif s == "equal_priority_oldest_first":
        tie = [
            msg(priority=MessagePriority.NORMAL, ts=T0 + timedelta(seconds=5)),
            msg(priority=MessagePriority.NORMAL, ts=T0),
        ]
        got = [m.timestamp.isoformat() for m in sorted(tie)]
        assert got == case["expected"]
        print("à priorité égale, le plus ancien d'abord :", [g[-8:] for g in got])
    elif s == "dict_round_trip_equal_and_correlated":
        rt = msg(MessageType.REQUEST, sender="agent-a", level=AgentLevel.STRATEGIC,
                 priority=MessagePriority.HIGH, content={"request_type": "get_analysis"},
                 recipient="agent-b", channel="hierarchical",
                 metadata={"conversation_id": "conv-test"},
                 ts=T0 + timedelta(microseconds=123456))
        rebuilt = Message.from_dict(rt.to_dict())
        assert (rebuilt == rt) == case["expected_equal"]
        assert rebuilt.is_response_to("nope") == case["expected_reply_correlation"]
        print("to_dict → from_dict : égalité", rebuilt == rt, "— reply_to non marqué sur une requête")
    elif s == "create_response_reversal":
        req = msg(MessageType.REQUEST, sender="strategic-1", level=AgentLevel.STRATEGIC,
                  content={"request_type": "get_status"}, recipient="tactical-1", mid="request-demo")
        resp = req.create_response({"status": "ok"})
        assert resp.sender == case["expected_sender"]
        assert resp.recipient == case["expected_recipient"]
        assert resp.type.value == case["expected_type"]
        assert resp.metadata["reply_to"] == case["expected_reply_to"]
        assert resp.is_response_to(req.id) == case["expected_is_response_to"]
        print(f"réponse : {resp.sender} → {resp.recipient}, type {resp.type.value}, reply_to={resp.metadata['reply_to'][:14]}…")
        print("  (create_response imprime une ligne de debug — un artefact réel du moteur)")
    elif s == "specialised_defaults":
        cmd = CommandMessage(sender="strategic-1", sender_level=AgentLevel.STRATEGIC,
                             command_type="execute_analysis", parameters={"depth": 2},
                             recipient="operational-1", timestamp=T0)
        evt = EventMessage(sender="tactical-1", sender_level=AgentLevel.TACTICAL,
                           event_type="resource_warning", description="cpu", details={}, timestamp=T0)
        info = InformationMessage(sender="tactical-1", sender_level=AgentLevel.TACTICAL,
                                  info_type="status_update", data={"progress": 0.5}, timestamp=T0)
        assert cmd.priority.value == case["command_priority"]
        assert cmd.requires_acknowledgement() == case["command_requires_ack"]
        assert sorted(cmd.content.keys()) == case["command_content_keys"]
        assert evt.recipient == case["event_recipient"]
        assert evt.priority.value == case["event_priority"]
        assert info.recipient == case["info_recipient"]
        print(f"COMMANDE : {cmd.priority.value}, ack requis = {cmd.requires_acknowledgement()}, contenu {sorted(cmd.content.keys())}")
        print(f"ÉVÉNEMENT : destinataire {evt.recipient} (broadcast), priorité {evt.priority.value}")

id du message COMMAND : command-895af07b  (préfixe « command »)
sorted() avec horodatages INVERSÉS : ['critical', 'high', 'normal', 'low']
à priorité égale, le plus ancien d'abord : ['12:00:00', '12:00:05']
to_dict → from_dict : égalité True — reply_to non marqué sur une requête
Created response response-8ac5c963 to request request-demo with reply_to=request-demo
réponse : tactical-1 → strategic-1, type response, reply_to=request-demo…
  (create_response imprime une ligne de debug — un artefact réel du moteur)
COMMANDE : high, ack requis = True, contenu ['command_type', 'parameters']
ÉVÉNEMENT : destinataire None (broadcast), priorité high


## 2. Le contrat de filtre — strict côté canal (#2161), muet côté topic

`Channel.matches_filter` honore **exactement cinq clés** (`message_type`,
`sender`, `priority`, `sender_level`, `content`). Depuis #2161, une clé hors
contrat **lève** au lieu d'être ignorée : l'ignorance silencieuse laissait
passer des filtres morts pour vivants. Le contrat parle en **chaînes** —
passer l'énuméré au lieu de la chaîne ne lève pas, il ne matche *jamais*.

Le matcher **du topic** (`Topic._matches_filter`, pub/sub) est une seconde
sémantique : il ne connaît pas `message_type` et ignore silencieusement les
clés inconnues. Les deux sont mesurées côte à côte.

In [3]:
local = LocalChannel("demo")
probe = msg(MessageType.COMMAND, content={"command_type": "execute_analysis"})

CRITERIA = {
    "scalar_type_match": {"message_type": "command"},
    "scalar_type_no_match": {"message_type": "event"},
    "list_type": {"message_type": ["command", "request"]},
    "enum_member_is_not_a_string": {"message_type": MessageType.COMMAND},
    "content_subkey": {"content": {"command_type": "execute_analysis"}},
    "content_subkey_no_match": {"content": {"command_type": "allocate_resources"}},
    "unknown_key_raises": {"destinataire": "tactical-1"},
    "sender_and_priority_combined": {"sender": "agent-a", "priority": "normal"},
}
TOPIC_CRITERIA = {
    "topic_has_no_message_type_key": {"message_type": "command"},
    "topic_silently_ignores_unknown_keys": {"nimporte": "quoi"},
    "topic_sender_still_honored": {"sender": "agent-b"},
}

for case in EXAMPLES["filter_cases"]:
    try:
        got = local.matches_filter(probe, CRITERIA[case["name"]])
        assert got == case["expected"], (case["name"], got, case["expected"])
        print(f"{case['name']:<32} -> {got}")
    except ValueError:
        assert case.get("raises") == "ValueError", case["name"]
        print(f"{case['name']:<32} -> ValueError (clé hors contrat, #2161)")
print()
topic_probe, topic_msg = Topic("demo"), msg(MessageType.EVENT, sender="agent-a")
for case in EXAMPLES["topic_filter_cases"]:
    got = topic_probe._matches_filter(topic_msg, TOPIC_CRITERIA[case["name"]])
    assert got == case["expected"], case["name"]
    print(f"Topic.{case['name']:<40} -> {got}")

scalar_type_match                -> True
scalar_type_no_match             -> False
list_type                        -> True
enum_member_is_not_a_string      -> False
content_subkey                   -> True
content_subkey_no_match          -> False
unknown_key_raises               -> ValueError (clé hors contrat, #2161)
sender_and_priority_combined     -> True

Topic.topic_has_no_message_type_key            -> True
Topic.topic_silently_ignores_unknown_keys      -> True
Topic.topic_sender_still_honored               -> False


## 3. Le middleware et sa table de routage — et le piège mesuré

`create_default_middleware()` est le câblage canonique (#1471) : un
`MessageMiddleware()` **nu** n'enregistre aucun canal et son bus est
silencieux. La table de routage est une fonction pure du message : COMMAND →
hiérarchique, INFORMATION → données *si* « analysis_result » dans l'info_type,
REQUEST → collaboration *si* « assistance », PUBLICATION → données, et les
types sans canal implémenté (EVENT, CONTROL) retombent sur le bus hiérarchique
(#1571 — les routes mortes sont soustraites, pas câblées sur des transports
vides).

Le piège : la table route « assistance » vers **collaboration**, mais le
câblage par défaut n'enregistre **pas** ce canal. Mesuré : l'envoi échoue
jusqu'à ce qu'on l'enregistre.

In [4]:
bare = MessageMiddleware()  # nu — aucun canal
assert bare.send_message(msg(MessageType.COMMAND, recipient="tactical-1")) == \
    EXAMPLES["middleware_cases"][0]["expected"]
print("middleware nu : send_message ->", False, "(#1471 : bus silencieux)")
print()

for case in EXAMPLES["routing_cases"]:
    content = {}
    if case["info_type"]:
        content["info_type"] = case["info_type"]
    if case["request_type"]:
        content["request_type"] = case["request_type"]
    m = msg(MessageType(case["message_type"]), content=content, channel=case["explicit_channel"])
    got = bare.determine_channel(m).value
    assert got == case["expected"], (case["message_type"], got, case["expected"])
    detail = case["info_type"] or case["request_type"] or (f"canal explicite={case['explicit_channel']}" if case["explicit_channel"] else "")
    print(f"{case['message_type']:<12} {detail:<28} -> {got}")
print()
mw = create_default_middleware()
ids = sorted(c.id for c in mw.channels.values())
assert ids == EXAMPLES["middleware_cases"][1]["expected"]
print("câblage canonique :", ids)
assistance = msg(MessageType.REQUEST, sender="tactical-1", level=AgentLevel.TACTICAL,
                 content={"request_type": "assistance"}, recipient="tactical-2")
gotcha = EXAMPLES["middleware_cases"][2]["expected"]
assert mw.send_message(assistance) == gotcha
print("« assistance » sur le middleware par défaut ->", gotcha, "(canal collaboration NON enregistré)")
mw.register_channel(CollaborationChannel(channel_id="collaboration_main"))
assistance2 = msg(MessageType.REQUEST, sender="tactical-1", level=AgentLevel.TACTICAL,
                  content={"request_type": "assistance"}, recipient="tactical-2")
assert mw.send_message(assistance2) == EXAMPLES["middleware_cases"][3]["expected"]
print("après enregistrement d'un CollaborationChannel  ->", True, "(table et câblage coïncident)")

middleware nu : send_message -> False (#1471 : bus silencieux)

command                                   -> hierarchical
information  status_update                -> hierarchical
information  analysis_result              -> data
request      get_analysis                 -> hierarchical
request      assistance                   -> collaboration
response                                  -> hierarchical
publication                               -> data
event                                     -> hierarchical
control                                   -> hierarchical
information  status_update                -> data

câblage canonique : ['data_main', 'hierarchical_main']
« assistance » sur le middleware par défaut -> False (canal collaboration NON enregistré)
après enregistrement d'un CollaborationChannel  -> True (table et câblage coïncident)


## 4. HierarchicalChannel — file à priorité, lecture NON destructive

Chaque destinataire a sa `PriorityQueue` : CRITICAL=0 sort avant NORMAL=2 avant
LOW=3, quel que soit l'ordre d'arrivée (miroir du §1). Pas de destinataire →
refus. Et surtout : `get_pending_messages` **ne consomme pas** la file —
contrairement au `LocalChannel` de référence qui la **draine**. Même méthode
abstraite, deux sémantiques opposées : mesuré, pas décrété.

In [5]:
hc = HierarchicalChannel("demo")
assert hc.send_message(msg(MessageType.COMMAND, recipient=None)) == EXAMPLES["hierarchical_cases"][0]["expected"]
print("sans destinataire ->", False)

for pr, off in [(MessagePriority.NORMAL, 0), (MessagePriority.CRITICAL, 5), (MessagePriority.LOW, 10)]:
    hc.send_message(msg(MessageType.COMMAND, sender="strategic-1", level=AgentLevel.STRATEGIC,
                        priority=pr, content={"command_type": "execute_analysis"},
                        recipient="tactical-1", ts=T0 + timedelta(seconds=off)))
order = [hc.receive_message("tactical-1", timeout=1.0).priority.value for _ in range(3)]
assert order == EXAMPLES["hierarchical_cases"][1]["expected"]
print("ordre de sortie de la file :", order)

hc2 = HierarchicalChannel("demo2")
for i in range(3):
    hc2.send_message(msg(MessageType.INFORMATION, recipient="tactical-1", ts=T0 + timedelta(seconds=i)))
pend = [len(hc2.get_pending_messages("tactical-1")) for _ in range(2)]
pend.append(hc2.clear_queue("tactical-1"))
pend.append(len(hc2.get_pending_messages("tactical-1")))
assert pend == EXAMPLES["hierarchical_cases"][2]["expected"]
print("pending x2 puis clear puis pending :", pend)

# la même méthode sur LocalChannel DRAINE — contraste mesuré
lc = LocalChannel("demo")
for i in range(2):
    lc.send_message(msg(MessageType.INFORMATION, recipient="tactical-1", ts=T0 + timedelta(seconds=i)))
drain = [len(lc.get_pending_messages("tactical-1")) for _ in range(2)]
PS = EXAMPLES["pending_semantics_cases"][0]
assert drain == PS["local"] and pend[:2] == PS["hierarchical"]
print("LocalChannel pending x2 :", drain, "(draine)  vs  Hierarchical :", PS["hierarchical"], "(préserve)")

hc3 = HierarchicalChannel("demo3")
hc3.send_message(msg(MessageType.COMMAND, sender="strategic-1", level=AgentLevel.STRATEGIC, recipient="tactical-9"))
hc3.send_message(msg(MessageType.INFORMATION, sender="operational-1", level=AgentLevel.OPERATIONAL, recipient="tactical-9"))
hc3.send_message(msg(MessageType.INFORMATION, sender="tactical-1", level=AgentLevel.TACTICAL, recipient="tactical-2"))
assert hc3.stats["by_direction"] == EXAMPLES["hierarchical_cases"][3]["expected"]
print("directions (préfixe du destinataire x niveau émetteur) :", hc3.stats["by_direction"])

sans destinataire -> False
ordre de sortie de la file : ['critical', 'normal', 'low']
pending x2 puis clear puis pending : [3, 3, 3, 0]
LocalChannel pending x2 : [2, 0] (draine)  vs  Hierarchical : [3, 3] (préserve)
directions (préfixe du destinataire x niveau émetteur) : {'strategic_to_tactical': 1, 'tactical_to_strategic': 0, 'tactical_to_operational': 0, 'operational_to_tactical': 1, 'same_level': 1}


## 5. CollaborationChannel — groupes fermés, historique partagé

Le canal horizontal gère des **groupes** (l'émetteur doit être membre — le
non-membre est refusé) et des **messages directs**. Deux sémantiques de
lecture cohabitent, mesurées : l'historique de groupe est un **journal
partagé** sans marque de lecture (on relit le même message ; ses propres
messages sont masqués), le message direct est **lu une fois** (drapeau).

In [6]:
cc = CollaborationChannel("demo")
cc.create_group(group_id="cellule-alpha", members=["sherlock", "watson"])
outsider = msg(MessageType.INFORMATION, sender="moriarty", level=AgentLevel.OPERATIONAL,
               content={"info_type": "status_update"}, metadata={"group_id": "cellule-alpha"})
member = msg(MessageType.INFORMATION, sender="sherlock", level=AgentLevel.OPERATIONAL,
             content={"info_type": "clue_found"}, metadata={"group_id": "cellule-alpha"},
             mid="info-clue-0001")
C0 = EXAMPLES["collaboration_cases"][0]
assert cc.send_message(outsider) == C0["outsider_send"]
assert cc.send_message(member) == C0["member_send"]
assert len(cc.get_group_messages("cellule-alpha")) == C0["history_len"]
print(f"non-membre -> {C0['outsider_send']} ; membre -> {C0['member_send']} ; historique = {C0['history_len']}")

C1 = EXAMPLES["collaboration_cases"][1]
r1, r2 = cc.receive_message("watson"), cc.receive_message("watson")
r3 = cc.receive_message("sherlock")
assert (r1.id if r1 else None) == C1["watson_first_read"]
assert (r2.id if r2 else None) == C1["watson_second_read"]
assert (r3.id if r3 else None) == C1["sherlock_read"]
print("Watson relit :", r2.id == r1.id, "| Sherlock (émetteur) reçoit :", r3 is None)

C2 = EXAMPLES["collaboration_cases"][2]
cc.send_message(msg(MessageType.INFORMATION, sender="mycroft", level=AgentLevel.STRATEGIC,
                    content={"info_type": "directive"}, recipient="lestrade", mid="info-direct-0001"))
d1, d2 = cc.receive_message("lestrade"), cc.receive_message("lestrade")
assert (d1.id if d1 else None) == C2["first"]
assert (d2.id if d2 else None) == C2["second"]
print("direct : 1re lecture =", d1.id[:16] + "…", "| 2e lecture =", d2)

non-membre -> False ; membre -> True ; historique = 1
Watson relit : True | Sherlock (émetteur) reçoit : True
direct : 1re lecture = info-direct-0001… | 2e lecture = None


## 6. DataChannel — le claim-check et le versionnement

Pour les charges volumineuses, le canal applique le patron **claim-check** :
au-delà de `max_inline_data_size` (10240 caractères), la charge est déposée
dans un `DataStore` **versionné** et le message voyage avec une référence
(`data_id` + `version_id`). À la réception, la référence est **résolue
transparentement** — le destinataire relit la charge complète sans voir la
mécanique. Chaque écriture crée une version ; on lit la dernière, ou une
version précise en l'épinglant.

In [7]:
dc = DataChannel("demo")
payload = {"paragraphs": ["analyse " * 40] * 60}  # > 10 k caractères
assert dc.max_inline_data_size == INV["max_inline_data_size"]
big = msg(MessageType.INFORMATION, sender="tactical-1", level=AgentLevel.TACTICAL,
          content={"info_type": "analysis_result", "data": payload}, recipient="strategic-1")
dc.send_message(big)
in_flight = dc.message_queues["strategic-1"][0]["message"].content
D0 = EXAMPLES["data_cases"][0]
assert (in_flight.get("data")) is D0["inline_none"] and (in_flight.get("data_reference") is not None) == D0["reference_present"]
print("en vol : data =", in_flight.get("data"), "| référence :", sorted(in_flight["data_reference"].keys()))

received = dc.receive_message("strategic-1")
D1 = EXAMPLES["data_cases"][1]
assert (received.content["data"] == payload) == D1["equal"]
assert ("data_reference" not in received.content) == D1["reference_gone"]
print("reçu : charge intégralement réhydratée =", received.content["data"] == payload,
      "| référence consommée =", "data_reference" not in received.content)

dc.store_data("corpus-notes", {"revision": 1}, metadata={"stage": "extraction"})
time.sleep(0.01)
dc.store_data("corpus-notes", {"revision": 2}, metadata={"stage": "synthesis"})
D2 = EXAMPLES["data_cases"][2]
versions = dc.data_store.get_versions("corpus-notes")
latest, latest_meta = dc.get_data("corpus-notes")
v1, _ = dc.get_data("corpus-notes", versions[0])
assert len(versions) == D2["versions_count"]
assert latest["revision"] == D2["latest_revision"] and v1["revision"] == D2["pinned_revision"]
print(f"versions : {len(versions)} | dernière = révision {latest['revision']} ({latest_meta['stage']})",
      f"| épinglée = révision {v1['revision']}")

en vol : data = None | référence : ['data_id', 'size', 'version_id']
reçu : charge intégralement réhydratée = True | référence consommée = True
versions : 2 | dernière = révision 2 (synthesis) | épinglée = révision 1


## 7. Pub/sub — historique pour abonnés tardifs, fan-out global

Le protocole gère des **topics** : historique borné (100) qui sert les abonnés
tardifs, livraison synchrone aux callbacks filtrés, et `publish` **rend la
liste des abonnés livrés**. Deux réparations historiques visibles au
comportement : chaque publication traverse aussi le middleware (#1500 — un
listener global reçoit les broadcasts, l'ancien `send_message` pour
`recipient=None` était mort), et **s'abonner n'émet plus de message** (#1571 —
l'ancien message SUBSCRIPTION routait vers un canal sans implémentation,
indélivrable, échec avalé : du théâtre, soustrait).

In [8]:
ps = None
try:
    ps = PublishSubscribeProtocol(mw)
    recus = []
    ps.subscribe(topic_id="analyses", subscriber_id="veille-1", callback=lambda m: recus.append(m.id))
    broadcasts = []
    mw.register_global_handler(lambda m: broadcasts.append(m.id))
    P0 = EXAMPLES["pubsub_cases"][0]
    recipients = ps.publish(topic_id="analyses", sender="tactical-1",
                            sender_level=AgentLevel.TACTICAL,
                            content={"info_type": "analysis_result"})
    assert recipients == P0["expected"] and len(recus) == P0["callback_hits"]
    print("publish rend les abonnés livrés :", recipients, "| callbacks appelés :", len(recus))

    before = len(broadcasts)
    ps.subscribe(topic_id="analyses", subscriber_id="veille-2")
    P2 = EXAMPLES["pubsub_cases"][2]
    assert (len(broadcasts) - before) == P2["expected"]
    print("s'abonner émet :", len(broadcasts) - before, "message (#1571 : l'émission théâtre est retirée)")

    for i in range(3):
        ps.publish(topic_id="analyses", sender="tactical-1", sender_level=AgentLevel.TACTICAL,
                   content={"info_type": "analysis_result", "seq": i})
    P1 = EXAMPLES["pubsub_cases"][1]
    assert (len(broadcasts) >= 4) == P1["expected"]
    print("listener global du middleware notifié :", len(broadcasts), "fois sur 4 publications (#1500)")

    late = ps.get_topic("analyses").get_recent_messages(2)
    P3 = EXAMPLES["pubsub_cases"][3]
    assert [m.content.get("seq") for m in late] == P3["expected"]
    print("abonné tardif, get_recent_messages(2) → seq", [m.content.get("seq") for m in late])
finally:
    ps.shutdown()  # le protocole spawn un thread de nettoyage : on l'arrête

publish rend les abonnés livrés : ['veille-1'] | callbacks appelés : 1
s'abonner émet : 0 message (#1571 : l'émission théâtre est retirée)
listener global du middleware notifié : 4 fois sur 4 publications (#1500)
abonné tardif, get_recent_messages(2) → seq [1, 2]


## 8. Requête-réponse — l'échec nommé, la corrélation, la réponse orpheline

`send_request` bloque jusqu'à réponse ou **lève** `RequestTimeoutError` — le
timeout ne rend jamais `None` silencieusement (#1355 : la course entre les
deux horloges masquait l'échec). La corrélation se fait par `reply_to`.
Et une réponse qui arrive sans requête en attente n'est **pas jetée** : elle
est parquée en « réponse anticipée » (5 minutes) — le mécanisme qui fait
survivre une réponse plus rapide que l'enregistrement de la requête.

In [9]:
rr = None
try:
    rr = RequestResponseProtocol(mw)
    R0 = EXAMPLES["request_response_cases"][0]
    try:
        rr.send_request(sender="strategic-1", sender_level=AgentLevel.STRATEGIC,
                        recipient="personne", request_type="get_analysis",
                        content={}, timeout=0.3)
        raise AssertionError("devait lever")
    except RequestTimeoutError:
        print(f"sans répondant -> {R0['expected']} (après ~{R0['elapsed_s']} s)")

    rid = rr.send_request_async_callback(sender="strategic-1", sender_level=AgentLevel.STRATEGIC,
                                         recipient="tactical-1", request_type="get_status",
                                         content={}, callback=lambda r, e: None, timeout=30)
    answer = msg(MessageType.RESPONSE, sender="tactical-1", level=AgentLevel.TACTICAL,
                 content={"status": "ok"}, recipient="strategic-1",
                 metadata={"reply_to": rid, "conversation_id": "conv-x"})
    R1 = EXAMPLES["request_response_cases"][1]
    assert rr.handle_response(answer) == R1["correlated"]
    assert rr.pending_requests[rid]["completed"].is_set() == R1["completed"]
    assert rr.pending_requests[rid]["response"].content["status"] == R1["stored"]
    print("handle_response(reply_to) :", R1["correlated"], "| complété :", R1["completed"], "| réponse rangée :", R1["stored"])

    orphan = msg(MessageType.RESPONSE, sender="tactical-2", level=AgentLevel.TACTICAL,
                 content={"status": "ko"}, recipient="strategic-1",
                 metadata={"reply_to": "request-inconnu"})
    R2 = EXAMPLES["request_response_cases"][2]
    assert rr.handle_response(orphan) == R2["handled"]
    assert len(rr.early_responses) == R2["early_count"]
    print("réponse orpheline : traitée =", R2["handled"], "| parquée en anticipée :", len(rr.early_responses))
finally:
    rr.shutdown()  # arrête le thread de surveillance des timeouts

sans répondant -> RequestTimeoutError (après ~0.31 s)
handle_response(reply_to) : True | complété : True | réponse rangée : ok
réponse orpheline : traitée = True | parquée en anticipée : 1


## À retenir

1. **La priorité est un ordre inversé** — `sorted()` sort CRITICAL en premier ;
   à priorité égale, FIFO. Les files hiérarchiques l'implémentent nativement.
2. **Le filtre de canal est strict** — cinq clés, des chaînes (pas les énumérés),
   une clé inconnue LÈVE (#2161). Le filtre de topic, lui, ignore silencieusement :
   deux sémantiques mesurées, ne pas les confondre.
3. **Toujours `create_default_middleware()`** — un middleware nu est un bus
   silencieux (#1471). Et la table route « assistance » vers un canal que le
   câblage par défaut n'enregistre pas : mesuré, corrigé par enregistrement.
4. **`get_pending_messages` n'a pas une sémantique unique** — LocalChannel
   draine, HierarchicalChannel préserve. Lire le canal, pas seulement l'interface.
5. **Les groupes sont fermés, l'historique est un journal** — non-membre refusé ;
   relecture possible côté groupe, lecture unique côté direct.
6. **Gros volumes = claim-check** — référence en vol, réhydratation transparente,
   versionnement en lecture (dernière ou épinglée).
7. **Les échecs sont nommés** — `RequestTimeoutError` lève au lieu de rendre
   None (#1355) ; une réponse orpheline est parquée, pas jetée ; les routes
   sans transport sont soustraites, pas théâtralisées (#1571).

*Asset CoursIA #1961 Phase 3 — les huit modules distillés vivent dans
`argumentation_analysis/core/communication/`. Contexte moteur récent :
#1471 (câblage canonique), #1500 (fan-out des broadcasts), #1571 (routes
mortes soustraites), #2161 (filtre strict), #1355 (timeout nommé).*